# TPC-H Customer Purchases Report

This notebook joins the TPC-H tables (`customer`, `nation`, `orders`) using the **Iara Python LazyFrame API**.

The execution is modularized across separate functions, returning:
- `customer_name`: Name of customer (`c_name`)
- `nation_name`: Nation name (`n_name`)
- `amount_of_purchases`: Number of orders placed (`count`)
- `total_purchase_value`: Total order purchase value (`sum(o_totalprice)`)
- `mktsegment`: Customer market segment (`c_mktsegment`)
- `account_balance`: Customer account balance (`c_acctbal`)

In [ ]:
import iara
from iara import functions as F

# Connect to workspace catalog
ws = iara.workspace()

In [ ]:
# Lazily scan / reference tables from the workspace catalog
customer = ws.table("customer")
nation = ws.table("nation")
orders = ws.table("orders")

In [ ]:
def join_customer_nation(customer: iara.LazyFrame, nation: iara.LazyFrame) -> iara.LazyFrame:
    """Enrich customer data with nation details on c_nationkey == n_nationkey."""
    return customer.join(
        nation,
        left_on="c_nationkey",
        right_on="n_nationkey",
        how="inner",
    )

customer_with_nation = join_customer_nation(customer, nation)

In [ ]:
def join_customer_orders(customer_nation: iara.LazyFrame, orders: iara.LazyFrame) -> iara.LazyFrame:
    """Join customer-nation records with their placed orders on c_custkey == o_custkey."""
    return customer_nation.join(
        orders,
        left_on="c_custkey",
        right_on="o_custkey",
        how="inner",
    )

customer_orders = join_customer_orders(customer_with_nation, orders)

In [ ]:
def aggregate_customer_purchases(joined: iara.LazyFrame) -> iara.LazyFrame:
    """Group by customer attributes and calculate purchase count and total value."""
    return joined.group_by(
        "c_custkey",
        "c_name",
        "n_name",
        "c_mktsegment",
        "c_acctbal",
    ).agg(
        amount_of_purchases=F.len(),
        total_purchase_value=F.col("o_totalprice").sum(),
    )

customer_purchases_agg = aggregate_customer_purchases(customer_orders)

In [ ]:
def format_customer_purchases_report(aggregated: iara.LazyFrame) -> iara.LazyFrame:
    """Select and alias requested fields, sorted by total purchase value descending."""
    return aggregated.select(
        customer_name=F.col("c_name"),
        nation_name=F.col("n_name"),
        amount_of_purchases=F.col("amount_of_purchases"),
        total_purchase_value=F.col("total_purchase_value"),
        mktsegment=F.col("c_mktsegment"),
        account_balance=F.col("c_acctbal"),
    ).sort("total_purchase_value", descending=True)

purchases_report = format_customer_purchases_report(customer_purchases_agg)

In [ ]:
def get_customer_purchases(
    customer: iara.LazyFrame,
    nation: iara.LazyFrame,
    orders: iara.LazyFrame,
) -> iara.LazyFrame:
    """Single composite pipeline function that joins TPC-H tables and builds the report."""
    cn = join_customer_nation(customer, nation)
    cno = join_customer_orders(cn, orders)
    agg = aggregate_customer_purchases(cno)
    return format_customer_purchases_report(agg)

In [ ]:
# Inspect the optimized logical execution plan and physical plan
print(purchases_report.explain())

In [ ]:
# In Iara Studio you can call purchases_report.preview() for the interactive table viewer,
# or collect() to materialize an Arrow/Iara DataFrame
df = purchases_report.collect()
df.show(15)